## Pipeline

In [1]:
import os
from pathlib import Path
import sys

# import numpy as np

# from osgeo import gdal
# from osgeo import gdal_array
# from osgeo import gdalconst
# from osgeo import ogr
# from osgeo import osr

# Set up the environment to point to the local repository.
nobackup = Path(os.environ['NOBACKUP'])
repoPath = nobackup / 'innovation-lab-repositories/lfm'

if not repoPath.exists():
    raise ValueError('Invalid path to repository.')
    
sys.path.append(str(repoPath))

from model.Pipeline import Pipeline


# ----------------------------------------------------------------------------
# getTileDbPath
# ----------------------------------------------------------------------------
def getTileDbPath(imageDir: Path, dbName: str = 'output_index.shp') -> Path:

    if not imageDir or not imageDir.exists() or not imageDir.is_dir():
        raise ValueError('A valid image directory must be provided.')

    fullPath: Path = imageDir / dbName

    if fullPath.exists():
        return fullPath

    MOON_SRS = 'GEOGCRS["Moon (2015) - Sphere / Ocentric", DATUM["Moon (2015) - Sphere", ELLIPSOID["Moon (2015) - Sphere",1737400,0, LENGTHUNIT["metre",1]]], PRIMEM["Reference Meridian",0, ANGLEUNIT["degree",0.0174532925199433]], CS[ellipsoidal,2], AXIS["geodetic latitude (Lat)",north, ORDER[1], ANGLEUNIT["degree",0.0174532925199433]], AXIS["geodetic longitude (Lon)",east, ORDER[2], ANGLEUNIT["degree",0.0174532925199433]], ID["IAU",30100,2015], REMARK["Source of IAU Coordinate systems: https://doi.org/10.1007/s10569-017-9805-5"]]'
    outFile: Path = imageDir / dbName
    
    # The database does not exist, so create it for the image directory.
    gdal.TileIndex(outFile, list(imageDir.glob('*.tif')), outputSRS=MOON_SRS)

    return outFile
      

In [2]:
outDir = Path('/explore/nobackup/people/rlgill/SystemTesting/lunar/cubes')
zone = '37S'
zoomLevel = 5

pipeline = Pipeline(outDir, zone, zoomLevel)


NameError: name 'Conversion' is not defined

In [21]:
OUT_DIR = Path('/explore/nobackup/people/rlgill/SystemTesting/lunar/cubes')
REPO_PATH = Path('/explore/nobackup/people/rlgill/innovation-lab-repositories/Armstrong_Benchmark')

# dbPath = Path('/explore/nobackup/projects/ilab/projects/Lunar_FM/data/testdata/NAC_PHO/NAC_PHO_E186N1213/WAC/output_index.shp')
dbPath = getTileDbPath(Path('/explore/nobackup/projects/ilab/projects/Lunar_FM/data/testdata/NAC_PHO/NAC_PHO_E186N1213/WAC/'))
zone = '37S'
zoomLevel = 5

cubeFile1: Path = pipeline(REPO_PATH, dbPath, OUT_DIR, zone, zoomLevel, (0, 0))
print('cubeFile1:', cubeFile1)

cubeFile2: Path = pipeline(REPO_PATH, dbPath, OUT_DIR, zone, zoomLevel, (9, 0))
print('cubeFile2:', cubeFile2)

cubeFile3: Path = pipeline(REPO_PATH, dbPath, OUT_DIR, zone, zoomLevel, (6, 10))
print('cubeFile3:', cubeFile3)

cubeFile4: Path = pipeline(REPO_PATH, dbPath, OUT_DIR, zone, 2, (9, 0))
print('cubeFile4:', cubeFile4)


NameError: name 'getTileDbPath' is not defined

## Include Other Image Types
- /explore/nobackup/projects/ilab/projects/Lunar_FM/data/testdata/Diviner/Hpar
- /explore/nobackup/projects/ilab/projects/Lunar_FM/data/testdata/gravity

In [3]:
dbPath = Path('/explore/nobackup/people/rlgill/SystemTesting/lunar/db.shp')
outDir = Path('/explore/nobackup/people/rlgill/SystemTesting/lunar/cubes/data2')

cubeFile5: Path = pipeline(REPO_PATH, dbPath, outDir, zone, zoomLevel, (0, 0))
print('cubeFile5:', cubeFile5)

cubeFile6: Path = pipeline(REPO_PATH, dbPath, outDir, zone, zoomLevel, (9, 0))
print('cubeFile6:', cubeFile6)


shape: (2, 512, 512)
cubeFile5: /explore/nobackup/people/rlgill/SystemTesting/lunar/cubes/data2/Cube-LTM37S-Zoom-5-Tile-0-0.tif
shape: (120, 512, 512)
cubeFile6: /explore/nobackup/people/rlgill/SystemTesting/lunar/cubes/data2/Cube-LTM37S-Zoom-5-Tile-9-0.tif


In [4]:
dbPath = Path('/explore/nobackup/people/rlgill/SystemTesting/lunar/db2.shp')
outDir = Path('/explore/nobackup/people/rlgill/SystemTesting/lunar/cubes/data3')

cubeFile7: Path = pipeline(REPO_PATH, dbPath, outDir, zone, zoomLevel, (0, 0))
print('cubeFile7:', cubeFile7)


shape: (4, 512, 512)
cubeFile7: /explore/nobackup/people/rlgill/SystemTesting/lunar/cubes/data3/Cube-LTM37S-Zoom-5-Tile-0-0.tif


## Process New Data
There is a new getTileDbPath that will create the necessary Shapefile database for the input image directory, if it does not already exist.

The area containing this data was given as 149.8 deg E/1.2 deg N.  We must convert that to a tile index.

#### To Do
- There might be extraneous coordinate conversions now that the user input is a coordinate, instead of a tile inded.
- Modify the pipeline to accept a lat/lon, instead of a tile index.

### LRO_NAC_Pho_Sites

In [19]:
# ---
# User input
# ---
imageDir = Path('/explore/nobackup/projects/lfm/processed_data/Lunar/LRO_NAC_Pho_Sites')
outDir = Path('/explore/nobackup/people/rlgill/SystemTesting/lunar/cubes/data4')
zone = '42N'
lon = 149.8
lat = 1.2

# ---
# Use the new function to retrieve the image database, creating it if necessary.
# ---
dbPath = getTileDbPath(imageDir)
print('dbPath:', dbPath)

# ---
# Find the tile index covering the point given above.  Note, this applies to the 
# zoom level defined in previous notebook cells.
# ---
tmsDir = REPO_PATH / 'TMS/RG'
tileDef: dict = getTileDef(tmsDir, zone, zoomLevel)

# Convert the input lat/lon to LTM.
easting, northing = latLonToLTM(lat, lon, tileDef)

# Get the tile index.
tileIndex = ltmToTileIndex(easting, northing, tileDef)
print('tileIndex:', tileIndex)

# ---
# Run the pipeline.
# ---
cubeFile8: Path = pipeline(REPO_PATH, dbPath, outDir, zone, zoomLevel, tileIndex)
print('cubeFile8:', cubeFile8)


dbPath: /explore/nobackup/projects/lfm/processed_data/Lunar/LRO_NAC_Pho_Sites/output_index.shp
tileIndex: (1, 63)
shape: (32, 512, 512)
cubeFile8: /explore/nobackup/people/rlgill/SystemTesting/lunar/cubes/data4/Cube-LTM42N-Zoom-5-Tile-1-63.tif


### LRO_WAC_Pho_Sites
This many files crashes GDAL when it attempts to create the output GeoTiff.

#### The following cell crashes.

In [ ]:
imageDir = Path('/explore/nobackup/projects/lfm/processed_data/Lunar/LRO_WAC_Pho_Sites')

dbPath = getTileDbPath(imageDir)
print('dbPath:', dbPath)

cubeFile9: Path = pipeline(REPO_PATH, dbPath, outDir, zone, zoomLevel, tileIndex)
print('cubeFile9:', cubeFile9)


### Mitigating the Crash
Create groups of twenty input files symbolically linked to the primary input directory.  This means we process many small batches.

This was accomplished manually, but could be automated in the pipeline.  The group size was chosen arbitrarily as a number that would definitely work.  This threshold can be changed.

In [18]:
imageDir = Path('/explore/nobackup/people/rlgill/SystemTesting/lunar/LRO_WAC_Pho_Sites')

for subdir in sorted(imageDir.iterdir()):

    if subdir.is_dir():
 
        print(f'Processing subdirectory: {subdir.name}')

        dbPath = getTileDbPath(subdir)
 
        # Make an output directory for this group.
        groupOutDir = outDir / subdir.stem
        groupOutDir.mkdir()
        
        # Note, this is using some variables set in previous notebook cells.
        cubeFile: Path = pipeline(REPO_PATH, dbPath, groupOutDir, zone, zoomLevel, tileIndex)


Processing subdirectory: group_00
shape: (70, 512, 512)
Processing subdirectory: group_01
shape: (70, 512, 512)
Processing subdirectory: group_02
shape: (70, 512, 512)
Processing subdirectory: group_03
shape: (70, 512, 512)
Processing subdirectory: group_04
shape: (70, 512, 512)
Processing subdirectory: group_05
shape: (70, 512, 512)
Processing subdirectory: group_06
shape: (70, 512, 512)
Processing subdirectory: group_07
shape: (70, 512, 512)
Processing subdirectory: group_08
shape: (70, 512, 512)
Processing subdirectory: group_09
shape: (70, 512, 512)
Processing subdirectory: group_10
shape: (70, 512, 512)
Processing subdirectory: group_11
shape: (70, 512, 512)
Processing subdirectory: group_12
shape: (70, 512, 512)
Processing subdirectory: group_13
shape: (70, 512, 512)
Processing subdirectory: group_14
shape: (70, 512, 512)
Processing subdirectory: group_15
shape: (70, 512, 512)
Processing subdirectory: group_16
shape: (70, 512, 512)
Processing subdirectory: group_17
shape: (70, 51